# 07 - Entrenamiento de Modelos ML

**Objetivo**: Entrenar Random Forest y XGBoost para predecir gems vs failures.

Pipeline:
1. Cargar features y labels
2. Preparar datos (merge, NaN handling, train/test split)
3. Entrenar Random Forest (baseline)
4. Entrenar XGBoost (precision)
5. Comparar resultados
6. Guardar modelos

### Metricas clave:
- **Precision (gems)**: Cuando dice "gem", que tan frecuente acierta
- **Recall (gems)**: De todas las gems reales, cuantas detecta
- **F1 Score**: Balance precision/recall
- **PR-AUC**: Mejor que ROC-AUC para datos desbalanceados

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from src.models.trainer import ModelTrainer
from src.models.evaluator import ModelEvaluator
import config

print("Modulos importados")

## 1. Cargar datos

In [ ]:
# Cargar features y labels
try:
    features_df = pd.read_parquet(config.PROCESSED_DIR / "features.parquet")
    labels_df = pd.read_parquet(config.PROCESSED_DIR / "labels.parquet")
    print(f"Features: {features_df.shape}")
    print(f"Labels: {labels_df.shape}")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Ejecuta notebooks 05 y 06 primero.")
    features_df = pd.DataFrame()
    labels_df = pd.DataFrame()

## 2. Entrenar modelos

In [ ]:
if not features_df.empty and not labels_df.empty:
    # Crear trainer
    trainer = ModelTrainer(random_seed=config.ML_CONFIG['random_seed'])
    
    # Entrenar ambos modelos (clasificacion binaria)
    results = trainer.train_all(
        features_df=features_df,
        labels_df=labels_df,
        target='label_binary',
    )
    
    print("\nEntrenamiento completado!")
    print(f"Modelos entrenados: {list(trainer.models.keys())}")
else:
    print("No hay datos suficientes para entrenar")

## 3. Evaluar modelos

In [ ]:
if 'results' in dir() and results:
    evaluator = ModelEvaluator()
    
    X_test = results['X_test']
    y_test = results['y_test']
    
    # Evaluar cada modelo
    all_eval_results = {}
    
    for name, model in trainer.models.items():
        print(f"\n{'='*50}")
        print(f"EVALUACION: {name.upper()}")
        print(f"{'='*50}")
        
        eval_result = evaluator.evaluate(model, X_test, y_test, model_name=name)
        all_eval_results[name] = eval_result
        
        print(f"\nAccuracy:  {eval_result['accuracy']:.3f}")
        print(f"Precision: {eval_result['precision']:.3f}")
        print(f"Recall:    {eval_result['recall']:.3f}")
        print(f"F1 Score:  {eval_result['f1']:.3f}")
        print(f"ROC-AUC:   {eval_result['roc_auc']:.3f}")
        print(f"PR-AUC:    {eval_result['pr_auc']:.3f}")
        print(f"\nClassification Report:")
        print(eval_result['classification_report'])

In [ ]:
# Confusion matrices
if 'all_eval_results' in dir() and all_eval_results:
    for name, eval_result in all_eval_results.items():
        evaluator.plot_confusion_matrix(
            y_test, eval_result['y_pred'], model_name=name
        )
        plt.show()

In [ ]:
# Curvas ROC
if 'all_eval_results' in dir() and all_eval_results:
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    for name, eval_result in all_eval_results.items():
        evaluator.plot_roc_curve(
            y_test, eval_result['y_prob'], model_name=name
        )
    plt.title('ROC Curves - Comparacion de Modelos')
    plt.show()

In [ ]:
# Curvas Precision-Recall (mas importante para datos desbalanceados)
if 'all_eval_results' in dir() and all_eval_results:
    for name, eval_result in all_eval_results.items():
        evaluator.plot_precision_recall_curve(
            y_test, eval_result['y_prob'], model_name=name
        )
    plt.title('Precision-Recall Curves')
    plt.show()

## 4. Comparacion de modelos

In [ ]:
if 'all_eval_results' in dir() and all_eval_results:
    comparison = evaluator.compare_models(all_eval_results)
    display(comparison)

## 5. Guardar modelos y resultados

In [ ]:
if 'trainer' in dir() and trainer.models:
    # Guardar modelos
    trainer.save_models(config.MODELS_DIR)
    print(f"Modelos guardados en: {config.MODELS_DIR}")
    
    # Guardar resultados de evaluacion como JSON
    eval_summary = {}
    for name, result in all_eval_results.items():
        eval_summary[name] = {
            'accuracy': result['accuracy'],
            'precision': result['precision'],
            'recall': result['recall'],
            'f1': result['f1'],
            'roc_auc': result['roc_auc'],
            'pr_auc': result['pr_auc'],
        }
    
    eval_path = config.MODELS_DIR / "evaluation_results.json"
    with open(eval_path, 'w') as f:
        json.dump(eval_summary, f, indent=2)
    print(f"Resultados guardados en: {eval_path}")
    
    # Guardar feature names
    feature_names = results.get('feature_names', [])
    if feature_names:
        with open(config.MODELS_DIR / "feature_names.json", 'w') as f:
            json.dump(feature_names, f)
    
    print("\nSiguiente paso: notebook 08_shap_analysis.ipynb")
else:
    print("No hay modelos para guardar")